In [ ]:
# Ô 1 – Kết nối Google Drive & Kiểm tra GPU Tesla
import os
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/EdgeAI'
os.makedirs(DRIVE, exist_ok=True)
print(f'✅ Đã kết nối Google Drive thành công tại: {DRIVE}')


In [ ]:
# Ô 2 – Nạp mã nguồn & giải nén dữ liệu hoàn toàn tự động (Zero-error)
%cd /content
import os, shutil, glob

# 1. Nạp mã nguồn EdgeAI
if not os.path.exists('/content/EdgeAI/src'):
    if os.path.exists(f'{DRIVE}/edgeai_code.zip'):
        print('-> Giải nén mã nguồn từ Drive: edgeai_code.zip')
        !mkdir -p /content/EdgeAI && unzip -q -o '$DRIVE/edgeai_code.zip' -d /content/EdgeAI
    else:
        print('-> Tải mã nguồn mới nhất từ GitHub...')
        !rm -rf /content/EdgeAI
        !git clone -b feat/v3-improvements https://github.com/LQLManZz/EdgeAI-DUT-Competition.git /content/EdgeAI 2>/dev/null || git clone https://github.com/LQLManZz/EdgeAI-DUT-Competition.git /content/EdgeAI
else:
    if os.path.exists(f'{DRIVE}/edgeai_code.zip'):
        !unzip -q -o '$DRIVE/edgeai_code.zip' -d /content/EdgeAI

# 2. Chuẩn bị tập dữ liệu (data/raw)
data_train = '/content/EdgeAI/data/raw/train/images'
if not os.path.exists(data_train) or len(glob.glob(data_train + '/*.png')) < 100:
    print('-> Đang chuẩn bị dữ liệu hình ảnh...')
    os.makedirs('/content/EdgeAI/data/raw', exist_ok=True)
    if os.path.exists(f'{DRIVE}/edgeai_data.zip'):
        print('-> Giải nén dữ liệu từ $DRIVE/edgeai_data.zip...')
        !unzip -q -o '$DRIVE/edgeai_data.zip' -d /content/EdgeAI
    elif os.path.exists(f'{DRIVE}/egde-ai-challenge-2026-lan-2-qualifiers.zip'):
        print('-> Giải nén từ $DRIVE/egde-ai-challenge-2026-lan-2-qualifiers.zip...')
        !unzip -q -o '$DRIVE/egde-ai-challenge-2026-lan-2-qualifiers.zip' -d /content/EdgeAI/data/raw
    elif os.path.exists('/content/egde-ai-challenge-2026-lan-2-qualifiers.zip'):
        print('-> Giải nén từ /content/egde-ai-challenge-2026-lan-2-qualifiers.zip...')
        !unzip -q -o '/content/egde-ai-challenge-2026-lan-2-qualifiers.zip' -d /content/EdgeAI/data/raw
    elif os.path.exists('/content/egde-ai-challenge-2026-lan-2-qualifiers/train'):
        print('-> Sao chép từ thư mục /content/egde-ai-challenge-2026-lan-2-qualifiers...')
        !cp -rn /content/egde-ai-challenge-2026-lan-2-qualifiers/* /content/EdgeAI/data/raw/
    else:
        if os.path.exists(f'{DRIVE}/kaggle.json'):
            !mkdir -p ~/.kaggle && cp '$DRIVE/kaggle.json' ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
        if os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
            print('-> Tải dataset từ Kaggle...')
            !kaggle competitions download -c egde-ai-challenge-2026-lan-2-qualifiers -p /content
            !unzip -q -o /content/egde-ai-challenge-2026-lan-2-qualifiers.zip -d /content/EdgeAI/data/raw
            !cp /content/egde-ai-challenge-2026-lan-2-qualifiers.zip '$DRIVE/' 2>/dev/null || true

# 3. Kiểm tra kiểm thử nhanh & in hàng đợi
%cd /content/EdgeAI
num_images = len(glob.glob('/content/EdgeAI/data/raw/train/images/*.png'))
print(f'-> Tổng số ảnh train tìm thấy: {num_images} ảnh')
!PYTHONPATH=src python -m pytest tests -q 2>&1 | tail -1
!cat configs/queue.json
print('\n✅ CẢ MÃ NGUỒN VÀ DỮ LIỆU ĐÃ HOÀN TẤT VÀ SẴN SÀNG HUẤN LUYỆN!')


In [ ]:
# Ô 3 – Chạy hàng đợi: train -> INT8 -> submission -> chẩn đoán INT8
%cd /content/EdgeAI
!python scripts/colab_run.py --drive '$DRIVE'


In [ ]:
# Ô 4 – Biểu đồ đường cong loss & val mAP50 của các thí nghiệm
%cd /content/EdgeAI
import json, os
import pandas as pd, matplotlib.pyplot as plt

queue_file = 'configs/queue.json'
if os.path.exists(queue_file):
    names = json.load(open(queue_file))['queue']
    for name in names:
        p = f'{DRIVE}/experiments/{name}/log.csv'
        if not os.path.exists(p):
            continue
        try:
            log = pd.read_csv(p)
            fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))
            fig.suptitle(name, fontsize=14, fontweight='bold')
            if 'head_hm_loss' in log.columns:
                log.plot(x='epoch', y='head_hm_loss', ax=ax[0], title='Heatmap loss (focal)', color='orange')
            if 'head_box_loss' in log.columns:
                log.plot(x='epoch', y='head_box_loss', ax=ax[1], title='Box loss (L1)', color='blue')
            if 'val_map50' in log.columns:
                sub = log.dropna(subset=['val_map50'])
                if not sub.empty:
                    sub.plot(x='epoch', y='val_map50', ax=ax[2], marker='o', title='val mAP50', color='green')
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f'Không thể vẽ biểu đồ cho {name}: {e}')


In [ ]:
# Ô 5 – Model Soup: Tự động gom trung bình trọng số các checkpoint tốt nhất để bứt phá mAP50
%cd /content/EdgeAI
import os, glob

soup_candidates = [
    f'{DRIVE}/experiments/h1p_kd_240x320/best.keras',
    f'{DRIVE}/experiments/h1p_kd_v2_240x320/best.keras'
]
existing_models = [m for m in soup_candidates if os.path.exists(m)]
print(f'Tìm thấy {len(existing_models)}/{len(soup_candidates)} mô hình ứng viên cho Model Soup:')
for m in existing_models:
    print('  ->', m)

if len(existing_models) >= 2:
    soup_dir = f'{DRIVE}/experiments/h1p_kd_soup_240x320'
    os.makedirs(soup_dir, exist_ok=True)
    soup_model = f'{soup_dir}/best.keras'
    print('\n[1/3] Đang trộn trọng số Model Soup...')
    models_args = ' '.join(f'"{m}"' for m in existing_models)
    !python scripts/model_soup.py --models {models_args} --out '{soup_model}'
    print('\n[2/3] Đang lượng tử hóa INT8 cho Model Soup...')
    !python -m edgedet.quant.export_int8 --model '{soup_model}' --data data/raw
    print('\n[3/3] Đang tạo file dự đoán submission...')
    !python -m edgedet.eval.make_submission --tflite '{soup_dir}/model_int8.tflite' --data data/raw --out '{soup_dir}/submission.csv'
    print(f'\n🎉 HOÀN TẤT MODEL SOUP! Kết quả đã lưu tại: {soup_dir}/submission.csv')
else:
    print('Cần ít nhất 2 mô hình đã hoàn thành để thực hiện Model Soup.')
